# Hamming Distance Analysis - Standalone Colab Notebook

Identifies sequence pairs differing by exactly N residues (default: 1),
filtered by temporal ordering. Produces a pairs table for downstream analysis.

**No installation required** - runs entirely on Colab's pre-installed packages.

**Steps:**
1. Setup (run the library cell)
2. Upload Data (CSV or FASTA)
3. Configure & Run Analysis
4. View Results
5. Download Results

---

In [ ]:
#@title 1. Setup & Library Code (run this cell first) { display-mode: "form" }

# =============================================================================
# IMPORTS
# =============================================================================

import datetime
import hashlib
import io
import re
from collections import defaultdict
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    ON_COLAB = True
except ImportError:
    ON_COLAB = False


# =============================================================================
# FASTA CLEANING FUNCTIONS (from fasta_cleaner.py)
# =============================================================================

CANONICAL_AA = set("ACDEFGHIKLMNPQRSTVWY")
DB_PREFIXES = {"sp", "tr", "gb", "ref", "emb", "dbj", "pir", "prf", "uniprot"}


def clean_sequence(sequence: str) -> str:
    """Clean an amino acid sequence: uppercase, replace non-canonical with _, remove non-letters."""
    sequence = sequence.upper().replace(" ", "").replace("\n", "").replace("\r", "")
    cleaned = ""
    for char in sequence:
        if char in CANONICAL_AA:
            cleaned += char
        elif char.isalpha():
            cleaned += "_"
    return cleaned


def hash_sequence(sequence: str) -> str:
    """Generate a unique ID for a sequence using SHA-256 (first 12 chars)."""
    return hashlib.sha256(sequence.encode()).hexdigest()[:12]


def parse_header(header: str) -> Dict[str, str]:
    """Parse FASTA header to extract metadata fields (name, date, etc.)."""
    header = header.lstrip(">")
    result = {"original_header": header, "name": "", "date": ""}

    date_pattern = re.compile(
        r"\b(\d{4}[-/]\d{1,2}[-/]\d{1,2}|\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\b"
    )

    delimiters = ["|", ";", "/", "\t"]
    fields = [header]
    for delim in delimiters:
        if delim in header:
            fields = [f.strip() for f in header.split(delim)]
            break

    extra_fields = []
    name_found = False
    for field in fields:
        field = field.strip()
        if not field:
            continue
        if field.lower() in DB_PREFIXES:
            continue
        if re.match(r"^[A-Z0-9]{4,12}$", field) and not name_found:
            extra_fields.append(field)
            continue
        date_match = date_pattern.search(field)
        if date_match and not result["date"]:
            result["date"] = date_match.group(1)
            if field == date_match.group(1):
                continue
        if not name_found:
            result["name"] = field
            name_found = True
        else:
            extra_fields.append(field)

    for i, field in enumerate(extra_fields, 1):
        result[f"field_{i}"] = field
    return result


def parse_fasta(content: str) -> List[Tuple[str, str]]:
    """Parse FASTA format content into list of (header, sequence) tuples."""
    sequences = []
    current_header = None
    current_seq = []
    for line in content.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if current_header is not None:
                sequences.append((current_header, "".join(current_seq)))
            current_header = line[1:]
            current_seq = []
        else:
            current_seq.append(line)
    if current_header is not None:
        sequences.append((current_header, "".join(current_seq)))
    return sequences


def handle_duplicate_metadata(metadata_list: List[Dict]) -> List[Dict]:
    """Append version markers (_v2, _v3) for same name+date with different sequences."""
    seen = defaultdict(list)
    for i, meta in enumerate(metadata_list):
        key = (meta.get("name", ""), meta.get("date", ""))
        seen[key].append(i)
    for key, indices in seen.items():
        if len(indices) > 1:
            seq_ids = [metadata_list[i]["sequence_id"] for i in indices]
            if len(set(seq_ids)) > 1:
                for version, idx in enumerate(indices, 1):
                    if version > 1:
                        original_name = metadata_list[idx].get("name", "")
                        metadata_list[idx]["name"] = f"{original_name}_v{version}"
    return metadata_list


def process_fasta_content(
    content: str, source_name: str = "input"
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Process FASTA content string and return (sequences_df, metadata_df)."""
    all_sequences = []
    all_metadata = []
    seen_sequences = {}

    parsed = parse_fasta(content)
    for header, raw_seq in parsed:
        cleaned = clean_sequence(raw_seq)
        if not cleaned:
            continue
        if cleaned in seen_sequences:
            seq_id = seen_sequences[cleaned]
        else:
            seq_id = hash_sequence(cleaned)
            seen_sequences[cleaned] = seq_id
            all_sequences.append({
                "sequence_id": seq_id, "sequence": cleaned, "length": len(cleaned)
            })
        meta = parse_header(header)
        meta["sequence_id"] = seq_id
        meta["source_file"] = source_name
        all_metadata.append(meta)

    all_metadata = handle_duplicate_metadata(all_metadata)
    sequences_df = pd.DataFrame(all_sequences)
    metadata_df = pd.DataFrame(all_metadata)
    if not metadata_df.empty:
        priority_cols = ["sequence_id", "original_header", "name", "date", "source_file"]
        other_cols = [c for c in metadata_df.columns if c not in priority_cols]
        metadata_df = metadata_df[priority_cols + other_cols]
    return sequences_df, metadata_df


# =============================================================================
# HAMMING DISTANCE FUNCTIONS (from hamming_lib.py)
# =============================================================================

DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%Y/%m/%d"]


def parse_date(date_str: str) -> Optional[datetime.date]:
    """Parse a date string trying multiple common formats."""
    if not date_str or not isinstance(date_str, str):
        return None
    date_str = date_str.strip()
    for fmt in DATE_FORMATS:
        try:
            return datetime.datetime.strptime(date_str, fmt).date()
        except ValueError:
            continue
    return None


def hamming_distance(seq_a: str, seq_b: str, max_distance: int = 1) -> int:
    """Compute Hamming distance with early exit once mismatches exceed max_distance."""
    if len(seq_a) != len(seq_b):
        raise ValueError(f"Sequences must be equal length: {len(seq_a)} vs {len(seq_b)}")
    mismatches = 0
    for a, b in zip(seq_a, seq_b):
        if a != b:
            mismatches += 1
            if mismatches > max_distance:
                return mismatches
    return mismatches


def find_mutation_position(seq_a: str, seq_b: str) -> Tuple[int, str, str]:
    """Find the single differing position between two Hamming-1 sequences."""
    for i, (a, b) in enumerate(zip(seq_a, seq_b)):
        if a != b:
            return (i + 1, a, b)
    raise ValueError("Sequences are identical -- no mutation found")


def _prepare_data(
    sequences_df: pd.DataFrame, metadata_df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Merge sequences with metadata and parse dates."""
    skipped_rows = []
    meta = metadata_df[["sequence_id", "name", "date"]].copy()
    meta["parsed_date"] = meta["date"].astype(str).apply(parse_date)

    no_date = meta[meta["parsed_date"].isna()]
    for _, row in no_date.iterrows():
        raw = str(row["date"])
        reason = "missing_date" if raw in ("", "nan", "None") else f"unparseable_date: {raw}"
        skipped_rows.append({
            "sequence_id": row["sequence_id"],
            "name": row.get("name", ""),
            "date_raw": raw,
            "skip_reason": reason,
        })

    has_date = meta[meta["parsed_date"].notna()].copy()
    has_date = has_date.sort_values("parsed_date").drop_duplicates(
        subset="sequence_id", keep="first"
    )
    merged = sequences_df.merge(has_date, on="sequence_id", how="inner")
    skipped_df = pd.DataFrame(
        skipped_rows, columns=["sequence_id", "name", "date_raw", "skip_reason"]
    )
    return merged, skipped_df


def find_hamming_pairs(
    sequences_df: pd.DataFrame,
    metadata_df: pd.DataFrame,
    distance: int = 1,
    progress_callback=None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Find all sequence pairs at exact Hamming distance, with temporal ordering."""
    merged, skipped_df = _prepare_data(sequences_df, metadata_df)

    if merged.empty:
        empty_results = pd.DataFrame(columns=[
            "seq_id_a", "name_a", "date_a",
            "seq_id_b", "name_b", "date_b",
            "sequence_length", "mutation_position",
            "residue_from", "residue_to",
            "hamming_distance", "same_date",
        ])
        return empty_results, skipped_df

    results = []
    groups = list(merged.groupby("length"))
    total_groups = len(groups)

    for group_idx, (length, group) in enumerate(groups):
        if progress_callback:
            progress_callback(group_idx, total_groups)

        if len(group) < 2:
            continue

        rows = group.to_dict("records")
        for rec_a, rec_b in combinations(rows, 2):
            dist = hamming_distance(
                rec_a["sequence"], rec_b["sequence"], max_distance=distance
            )
            if dist != distance:
                continue

            date_a = rec_a["parsed_date"]
            date_b = rec_b["parsed_date"]
            same_date = date_a == date_b

            if date_a > date_b:
                rec_a, rec_b = rec_b, rec_a
                date_a, date_b = date_b, date_a

            if distance == 1:
                pos, res_from, res_to = find_mutation_position(
                    rec_a["sequence"], rec_b["sequence"]
                )
            else:
                pos, res_from, res_to = None, None, None

            results.append({
                "seq_id_a": rec_a["sequence_id"],
                "name_a": rec_a["name"],
                "date_a": str(date_a),
                "seq_id_b": rec_b["sequence_id"],
                "name_b": rec_b["name"],
                "date_b": str(date_b),
                "sequence_length": length,
                "mutation_position": pos,
                "residue_from": res_from,
                "residue_to": res_to,
                "hamming_distance": dist,
                "same_date": same_date,
            })

    if progress_callback:
        progress_callback(total_groups, total_groups)

    results_df = pd.DataFrame(results, columns=[
        "seq_id_a", "name_a", "date_a",
        "seq_id_b", "name_b", "date_b",
        "sequence_length", "mutation_position",
        "residue_from", "residue_to",
        "hamming_distance", "same_date",
    ])
    return results_df, skipped_df


def find_hamming_pairs_from_csv(
    sequences_path, metadata_path, distance: int = 1
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Find Hamming pairs from CSV file paths."""
    sequences_df = pd.read_csv(sequences_path)
    metadata_df = pd.read_csv(metadata_path)
    return find_hamming_pairs(sequences_df, metadata_df, distance=distance)


def save_hamming_results(
    results_df: pd.DataFrame,
    skipped_df: pd.DataFrame,
    output_dir=".",
    prefix: str = "",
) -> Tuple[Path, Path]:
    """Save Hamming analysis results to CSV files."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    results_path = output_dir / f"{prefix}hamming_results.csv"
    skipped_path = output_dir / f"{prefix}hamming_skipped.csv"
    results_df.to_csv(results_path, index=False)
    skipped_df.to_csv(skipped_path, index=False)
    return results_path, skipped_path


# =============================================================================
print("Setup complete -- all library functions loaded.")

---
## Step 2: Upload Data

Choose one of three tabs below: upload pre-made CSVs, upload FASTA files, or paste FASTA text directly.

In [ ]:
sequences_df = None
metadata_df = None

upload_output = widgets.Output()

# --- Option 1: Upload CSVs ---
csv_btn = widgets.Button(description="Upload CSVs", button_style="primary",
                         tooltip="Upload sequences.csv and metadata.csv")

def on_csv_click(btn):
    global sequences_df, metadata_df
    with upload_output:
        clear_output()
        try:
            if not ON_COLAB:
                print("CSV upload requires Google Colab. Use 'Paste FASTA' instead.")
                return

            print("Select your sequences CSV file...")
            uploaded_seq = colab_files.upload()
            if not uploaded_seq:
                print("No file selected.")
                return
            fname, content = next(iter(uploaded_seq.items()))
            sequences_df = pd.read_csv(io.BytesIO(content))
            print(f"Loaded sequences from: {fname}")

            print("\nSelect your metadata CSV file...")
            uploaded_meta = colab_files.upload()
            if not uploaded_meta:
                print("No file selected.")
                return
            fname, content = next(iter(uploaded_meta.items()))
            metadata_df = pd.read_csv(io.BytesIO(content))
            print(f"Loaded metadata from: {fname}")

            # Validate columns
            req_seq = {"sequence_id", "sequence", "length"}
            req_meta = {"sequence_id", "name", "date"}
            missing_seq = req_seq - set(sequences_df.columns)
            missing_meta = req_meta - set(metadata_df.columns)
            if missing_seq:
                print(f"\nSequences CSV missing columns: {missing_seq}")
                print(f"Found columns: {list(sequences_df.columns)}")
                sequences_df = None
                return
            if missing_meta:
                print(f"\nMetadata CSV missing columns: {missing_meta}")
                print(f"Found columns: {list(metadata_df.columns)}")
                metadata_df = None
                return

            print(f"\nLoaded {len(sequences_df)} sequences, {len(metadata_df)} metadata records")
            display(sequences_df.head())
            display(metadata_df.head())
        except Exception as e:
            print(f"Error: {e}")

csv_btn.on_click(on_csv_click)

# --- Option 2: Upload FASTA ---
fasta_btn = widgets.Button(description="Upload FASTA", button_style="primary",
                           tooltip="Upload one or more .fasta files")

def on_fasta_click(btn):
    global sequences_df, metadata_df
    with upload_output:
        clear_output()
        try:
            if not ON_COLAB:
                print("File upload requires Google Colab. Use 'Paste FASTA' instead.")
                return

            print("Select FASTA file(s)...")
            uploaded = colab_files.upload()
            if not uploaded:
                print("No files selected.")
                return

            all_seqs, all_meta = [], []
            for fname, content in uploaded.items():
                text = content.decode("utf-8")
                s, m = process_fasta_content(text, fname)
                all_seqs.append(s)
                all_meta.append(m)
                print(f"  {fname}: {len(s)} sequences")

            sequences_df = pd.concat(all_seqs, ignore_index=True).drop_duplicates(
                subset="sequence_id", keep="first"
            )
            metadata_df = pd.concat(all_meta, ignore_index=True)

            print(f"\nProcessed {len(sequences_df)} unique sequences from {len(all_seqs)} file(s)")
            print(f"Metadata records: {len(metadata_df)}")
            display(sequences_df.head())
            display(metadata_df.head())
        except Exception as e:
            print(f"Error: {e}")

fasta_btn.on_click(on_fasta_click)

# --- Option 3: Paste FASTA ---
fasta_text = widgets.Textarea(
    placeholder=">seq1|name1|2024-01-15\nMKTAYIALSY...\n>seq2|name2|2024-03-20\nMKTAYIALSL...",
    layout=widgets.Layout(width="100%", height="200px"),
    description="FASTA:",
)
paste_btn = widgets.Button(description="Process Text", button_style="primary")

def on_paste_click(btn):
    global sequences_df, metadata_df
    with upload_output:
        clear_output()
        text = fasta_text.value.strip()
        if not text:
            print("Please paste FASTA text above first.")
            return
        try:
            sequences_df, metadata_df = process_fasta_content(text, "pasted_input")
            print(f"Processed {len(sequences_df)} unique sequences")
            print(f"Metadata records: {len(metadata_df)}")
            display(sequences_df.head())
            display(metadata_df.head())
        except Exception as e:
            print(f"Error: {e}")

paste_btn.on_click(on_paste_click)

# --- Display ---
display(HTML("<h3>Choose an upload method:</h3>"))
display(widgets.HBox([csv_btn, fasta_btn]))
display(HTML("<b>Or paste FASTA text directly:</b>"))
display(fasta_text)
display(paste_btn)
display(HTML("<hr>"))
display(upload_output)

---
## Step 3: Configure & Run Analysis

In [ ]:
results_df = None
skipped_df = None

distance_slider = widgets.IntSlider(
    value=1, min=1, max=10, step=1, description="Hamming distance:",
    style={"description_width": "120px"},
)
run_btn = widgets.Button(description="Run Analysis", button_style="success")
progress = widgets.IntProgress(min=0, max=100, description="Progress:", bar_style="info")
run_output = widgets.Output()

def on_run_click(btn):
    global results_df, skipped_df
    with run_output:
        clear_output()
        if sequences_df is None or metadata_df is None:
            print("Please upload data first (Step 2).")
            return

        dist = distance_slider.value
        n_seq = len(sequences_df)
        print(f"Running Hamming-{dist} analysis on {n_seq} sequences...")

        if n_seq > 5000:
            print(f"Note: {n_seq} sequences will produce many pairwise comparisons. This may take a while.")

        progress.value = 0
        progress.max = 100

        def update_progress(current, total):
            if total > 0:
                progress.max = total
                progress.value = current

        results_df, skipped_df = find_hamming_pairs(
            sequences_df, metadata_df,
            distance=dist,
            progress_callback=update_progress,
        )

        print(f"\nResults:")
        print(f"  {len(results_df)} Hamming-{dist} pairs found")
        print(f"  {len(skipped_df)} records skipped")

        if not results_df.empty:
            same_count = int(results_df["same_date"].sum())
            directed_count = len(results_df) - same_count
            print(f"  {directed_count} temporally directed pairs")
            print(f"  {same_count} same-date pairs")

        if not skipped_df.empty:
            print(f"\nSkipped breakdown:")
            for reason, count in skipped_df["skip_reason"].value_counts().items():
                print(f"  {reason}: {count}")

run_btn.on_click(on_run_click)

display(distance_slider)
display(widgets.HBox([run_btn, progress]))
display(run_output)

---
## Step 4: View Results

In [ ]:
if results_df is None:
    print("Run the analysis first (Step 3).")
elif results_df.empty:
    print("No pairs found at the specified Hamming distance.")
    if skipped_df is not None and not skipped_df.empty:
        print(f"\n{len(skipped_df)} records were skipped:")
        display(skipped_df)
else:
    print(f"Showing first 20 of {len(results_df)} pairs:\n")
    display(results_df.head(20))

    # Mutation summary for Hamming-1
    if "residue_from" in results_df.columns and results_df["residue_from"].notna().any():
        print("\n--- Residue Substitution Frequencies ---")
        sub_counts = (
            results_df
            .dropna(subset=["residue_from", "residue_to"])
            .groupby(["residue_from", "residue_to"])
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        display(sub_counts.head(20))

        print("\n--- Most Common Mutation Positions ---")
        pos_counts = (
            results_df
            .dropna(subset=["mutation_position"])
            ["mutation_position"]
            .astype(int)
            .value_counts()
            .head(10)
        )
        display(pos_counts.to_frame("count"))

    if skipped_df is not None and not skipped_df.empty:
        print(f"\n--- Skipped Records ({len(skipped_df)}) ---")
        display(skipped_df)

---
## Step 5: Download Results

In [ ]:
prefix_input = widgets.Text(
    value="", placeholder="optional filename prefix", description="Prefix:"
)
download_btn = widgets.Button(description="Download Results", button_style="primary")
download_output = widgets.Output()

def on_download_click(btn):
    with download_output:
        clear_output()
        if results_df is None:
            print("Run the analysis first (Step 3).")
            return

        prefix = prefix_input.value.strip()
        if prefix and not prefix.endswith("_"):
            prefix += "_"

        output_dir = Path("/tmp/hamming_output")
        results_path, skipped_path = save_hamming_results(
            results_df, skipped_df, output_dir=output_dir, prefix=prefix
        )

        print(f"Saved: {results_path} ({len(results_df)} rows)")
        print(f"Saved: {skipped_path} ({len(skipped_df)} rows)")

        if ON_COLAB:
            colab_files.download(str(results_path))
            if not skipped_df.empty:
                colab_files.download(str(skipped_path))
            print("\nDownload started.")
        else:
            print(f"\nNot on Colab -- files saved to {output_dir}")

download_btn.on_click(on_download_click)

display(prefix_input)
display(download_btn)
display(download_output)